In [8]:
def multi_goal_maze(maze, start):
    rows = len(maze)
    cols = len(maze[0])
    goals = []
    for i in range(rows):
        for j in range(cols):
            if maze[i][j] == 'G':
                goals.append((i, j))
    goal_index = {goals[i]: i for i in range(len(goals))}
    target_mask = (1 << len(goals)) - 1

    def heuristic(x, y, mask):
        if mask == target_mask:
            return 0
        d = []
        for i in range(len(goals)):
            if not (mask & (1 << i)):
                gx, gy = goals[i]
                d.append(abs(x - gx) + abs(y - gy))
        return min(d) if d else 0

    open_list = [(heuristic(start[0], start[1], 0), 0, start[0], start[1], 0, [start])]
    visited = {}

    while open_list:
        open_list.sort()
        f, g, x, y, mask, path = open_list.pop(0)
        if (x, y, mask) in visited and visited[(x, y, mask)] <= g:
            continue
        visited[(x, y, mask)] = g
        if mask == target_mask:
            return path
        for dx, dy in [(1,0),(-1,0),(0,1),(0,-1)]:
            nx, ny = x + dx, y + dy
            if 0 <= nx < rows and 0 <= ny < cols and maze[nx][ny] != '#':
                new_mask = mask
                if (nx, ny) in goal_index:
                    new_mask |= 1 << goal_index[(nx, ny)]
                new_g = g + 1
                new_f = new_g + heuristic(nx, ny, new_mask)
                open_list.append((new_f, new_g, nx, ny, new_mask, path + [(nx, ny)]))
    return None
maze = [
    ['S', '.', '.', '#', '.', '.', 'G'],
    ['.', '#', '.', '#', '.', '#', '.'],
    ['.', '#', '.', '.', '.', '#', '.'],
    ['.', '.', '#', '#', '.', '.', '.'],
    ['G', '.', '.', '.', '#', '.', 'G']
]

start = (0, 0)

result = multi_goal_maze(maze, start)
print(result)


[(0, 0), (1, 0), (2, 0), (3, 0), (4, 0), (3, 0), (2, 0), (1, 0), (0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (2, 3), (2, 4), (3, 4), (3, 5), (3, 6), (4, 6), (3, 6), (2, 6), (1, 6), (0, 6)]


In [9]:
def dynamic_a_star(graph, start, goal, cost_changes):
    open_list = [(0, 0, start, [start])]
    g_score = {start: 0}
    closed = set()

    while open_list:
        open_list.sort()
        f, g, node, path = open_list.pop(0)
        if node in closed:
            continue
        if node == goal:
            return path
        closed.add(node)
        if node in cost_changes:
            for u, v, new_cost in cost_changes[node]:
                graph[u][v] = new_cost
        for neighbor in graph[node]:
            tentative_g = g + graph[node][neighbor]
            if neighbor not in g_score or tentative_g < g_score[neighbor]:
                g_score[neighbor] = tentative_g
                h = 0
                open_list.append((tentative_g + h, tentative_g, neighbor, path + [neighbor]))
                if neighbor in closed:
                    closed.remove(neighbor)
    return None
graph = {
    'A': {'B': 2, 'C': 5},
    'B': {'C': 1, 'D': 4},
    'C': {'D': 1, 'E': 7},
    'D': {'E': 3},
    'E': {}
}

cost_changes = {
    'B': [('B', 'D', 1)],
    'C': [('C', 'E', 2)]
}

start = 'A'
goal = 'E'

result = dynamic_a_star(graph, start, goal, cost_changes)
print(result)



['A', 'B', 'C', 'E']


In [12]:
def delivery_route(points, start):
    current = start
    time = 0
    route = [start]
    remaining = points[:]

    while remaining:
        best = None
        best_score = float('inf')
        for p in remaining:
            dist = abs(current[0] - p[0]) + abs(current[1] - p[1])
            arrival = time + dist
            start_w, end_w = p[2]
            if arrival > end_w:
                continue
            wait = max(0, start_w - arrival)
            urgency = end_w - arrival
            score = urgency + dist + wait
            if score < best_score:
                best_score = score
                best = p
        if not best:
            return None
        dist = abs(current[0] - best[0]) + abs(current[1] - best[1])
        time += dist
        if time < best[2][0]:
            time = best[2][0]
        current = (best[0], best[1])
        route.append(current)
        remaining.remove(best)
    return route
start = (0, 0)

points = [
    (2, 2, (0, 15)),
    (4, 1, (3, 20)),
    (6, 3, (5, 25)),
    (3, 5, (2, 18))
]

result = delivery_route(points, start)
print(result)



[(0, 0), (2, 2), (3, 5), (4, 1), (6, 3)]
